<a href="https://colab.research.google.com/github/Sanath-cmd/Internship_ITT/blob/main/Algorithms/Fashion_MNIST_ANN_hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [7]:
torch.manual_seed= 42

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'The device in use {device}')

The device in use cuda


In [9]:
train_data = pd.read_csv('/content/drive/MyDrive/fashion-mnist_train.csv')
test_data = pd.read_csv('/content/drive/MyDrive/fashion-mnist_test.csv')

In [10]:
X_train, y_train = train_data.iloc[:, 1:].values, train_data.iloc[:, 0].values
X_test, y_test = test_data.iloc[:, 1:].values, test_data.iloc[:, 0].values

In [11]:
X_train = X_train/255
X_test = X_test/255

In [12]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype= torch.float32)
    self.labels = torch.tensor(labels, dtype= torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [13]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)


In [14]:
print(y_train[:20])

[2 9 6 0 3 4 4 5 4 8 0 8 9 0 2 2 9 3 3 3]


In [15]:
class Model(nn.Module):
  def __init__(self, input_size, output_size, num_hidden_layers, neuron_per_layer, dropout_prob):
    super().__init__()
    layers = []
    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_size, neuron_per_layer)) # Corrected line
      layers.append(nn.BatchNorm1d(neuron_per_layer))
      layers.append(nn.ReLU()) # Corrected ReLU
      layers.append(nn.Dropout(dropout_prob))
      input_size = neuron_per_layer

    layers.append(nn.Linear(neuron_per_layer, output_size))
    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

In [16]:
def objective(trial):
  num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 10)
  neuron_per_layer = trial.suggest_int("neuron_per_layer", 8, 256, step= 8)
  epochs = trial.suggest_int('epochs', 30, 70, step= 10)
  learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log= True)
  dropout_prob = trial.suggest_float('dropout_prob', 0.1, 0.5, step= 0.1)
  batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
  train_loader = DataLoader(train_dataset, batch_size= batch_size, shuffle = True, pin_memory= True)
  test_loader = DataLoader(test_dataset, batch_size= batch_size, shuffle = False, pin_memory= True)
  optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log= True)

  input_size = 784
  output_size = 10

  model = Model(input_size, output_size, num_hidden_layers, neuron_per_layer, dropout_prob)
  model = model.to(device)

  criterion = nn.CrossEntropyLoss()
  if optimizer_name =='SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr= learning_rate, weight_decay= weight_decay)
  elif optimizer_name == 'Adam':
    optimizer = torch.optim.Adam(model.parameters(), lr= learning_rate, weight_decay= weight_decay)
  else:
    optimizer = torch.optim.RMSprop(model.parameters(), lr= learning_rate, weight_decay= weight_decay)


  for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
      output = model(batch_features)
      loss = criterion(output, batch_labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
      for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        output = model(batch_features)
        _ , predicted = torch.max(output.data, 1)
        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()
    accuracy = (correct/total)

  return accuracy



In [17]:
!pip install optuna


In [18]:
import optuna

study = optuna.create_study(direction= 'maximize')


[I 2026-03-11 04:43:52,131] A new study created in memory with name: no-name-168cade7-a02e-4835-929c-6a2a0ec1e53e


In [ ]:
study.optimize(objective, n_trials= 50)


[I 2026-03-11 04:50:50,766] Trial 0 finished with value: 0.8983 and parameters: {'num_hidden_layers': 8, 'neuron_per_layer': 144, 'epochs': 70, 'learning_rate': 0.00023162098151956623, 'dropout_prob': 0.30000000000000004, 'batch_size': 64, 'optimizer': 'RMSprop', 'weight_decay': 2.3054290096121045e-05}. Best is trial 0 with value: 0.8983.
[I 2026-03-11 04:52:55,162] Trial 1 finished with value: 0.8457 and parameters: {'num_hidden_layers': 3, 'neuron_per_layer': 112, 'epochs': 50, 'learning_rate': 0.00014823321818154626, 'dropout_prob': 0.30000000000000004, 'batch_size': 128, 'optimizer': 'SGD', 'weight_decay': 0.007621149321645288}. Best is trial 0 with value: 0.8983.
[I 2026-03-11 04:57:35,643] Trial 2 finished with value: 0.873 and parameters: {'num_hidden_layers': 5, 'neuron_per_layer': 88, 'epochs': 60, 'learning_rate': 0.0011534483905677267, 'dropout_prob': 0.2, 'batch_size': 64, 'optimizer': 'RMSprop', 'weight_decay': 0.0012805363943620327}. Best is trial 0 with value: 0.8983.
[I